# Mahalanobis Distance Matching
This script is an alternative to `match_cells.ipynb`. It matches treatment and control cells using Mahalanobis Distance Matching (MDM) rather than PSM. The script contains covariate balance diagnostics for direct comparison to PSM.

In [ ]:
# Select PA
site_id = 2017

In [ ]:
from pathlib import Path
import sys
import os
import ee
import geemap

cur = Path.cwd().resolve()
for parent in [cur] + list(cur.parents):
    if parent.name == "tpae":
        os.chdir(parent)
        break

sys.path.insert(0, str((Path.cwd() / "src").resolve()))

from utils.variables import (
    PROJECT,
    EE_CRS_METERS,
    PSM_CELL_SIZE,
    BIOME_ASSET_ID,
    HGFC_ASSET_ID,
)

from absolute_effectiveness.site_selector import SiteSelector
from psm.prepare_pa_grid import load_pa_candidate_cells
from psm.covariates import build_resampled_covariates
from psm.cell_features import extract_cells_with_covariates
from psm.match_cells import (
    match_treatment_control_mdm,
    filter_matched_grids,
    save_matching_outputs,
)
from psm.diagnostics import pair_covariate_balance

ee.Authenticate()
ee.Initialize(project=PROJECT)

site_selector = SiteSelector()

EE_CRS_1km = ee.Projection(EE_CRS_METERS).atScale(PSM_CELL_SIZE)

## Data Prep
Load candidate cells and covariates.

In [ ]:
pa_ctx = load_pa_candidate_cells(site_id, site_selector)
PA_ID = pa_ctx["PA_ID"]
test_sites = pa_ctx["test_sites"]
site_geom = pa_ctx["site_geom"]
treatment_cells = pa_ctx["treatment_cells"]
control_cells = pa_ctx["control_cells"]
grid_fc = pa_ctx["grid_fc"]
covariates = build_resampled_covariates(EE_CRS_1km)

## Match Cells

In [ ]:
# Aggregate covariates within grid cells
grid_fc, cells_df = extract_cells_with_covariates(grid_fc, covariates, EE_CRS_1km)

In [ ]:
# Nearest neighbor matching by Mahalanobis distance,
# exact match on country and ecoregion (with biome as a fallback)
match_df, treat_df, control_df = match_treatment_control_mdm(cells_df)

In [ ]:
# Save matched cells
matched_grids = filter_matched_grids(grid_fc, match_df)
# save_matching_outputs(matched_grids, match_df, PA_ID)

## Diagnostics

In [ ]:
# Pair-level covariate balance (Feng et al. 2022).
# Better metric of per-match covariate balance, but doesn't have a before/after comparison.
results_df = pair_covariate_balance(match_df, cells_df)
# results_df.to_csv(f'results/MDM_balance_{site_id}.csv', index=False)

## Visualization

In [ ]:
# Visualize covariates

Map = geemap.Map()
Map.add_basemap("CartoDB.DarkMatter")
Map.centerObject(grid_fc)

Map.addLayer(
    covariates.select("elevation"),
    {"min": 85, "max": 4200, "palette": ["blue", "green", "yellow", "red"]},
    "Elevation", 0
)
Map.addLayer(
    covariates.select("slope"),
    {"min": 0, "max": 24, "palette": ["blue", "green", "yellow", "red"]},
    "Slope", 0
)
Map.addLayer(
    covariates.select("treecover2000"),
    {"min": 0, "max": 100, "palette": ["white", "green"]},
    "Tree Cover (2000)", 0
)
Map.addLayer(
    covariates.select("travel_time"),
    {"min": 56, "max": 2731, "palette": ["blue", "green", "yellow", "red"]},
    "Travel Time (2015)", 0
)
Map.addLayer(
    covariates.select("log_pop_density"),
    {"min": 0, "max": 5, "palette": ["blue", "green", "yellow", "red"]},
    "Population Density (2000)", 0
)
Map.addLayer(
    covariates.select("human_footprint"),
    {"min": 0, "max": 44, "palette": ["blue", "green", "yellow", "red"]},
    "Human Footprint (1993)", 0
)
Map.addLayer(
    covariates.select("ag_suitability"),
    {"min": 200, "max": 9800, "palette": ["blue", "green", "yellow", "red"]},
    "Agricultural Suitability (2001-2020)", 0
)

Map

In [ ]:
# Visualize matched cells

Map = geemap.Map()
# Map.add_basemap("CartoDB.Positron")
Map.add_basemap("CartoDB.DarkMatter")
ecoRegions = ee.FeatureCollection(BIOME_ASSET_ID)

color_updates = [
    {"ECO_ID": 204, "COLOR": '#B3493B'},
    {"ECO_ID": 245, "COLOR": '#267400'},
    {"ECO_ID": 259, "COLOR": '#004600'},
    {"ECO_ID": 286, "COLOR": '#82F178'},
    {"ECO_ID": 316, "COLOR": '#E600AA'},
    {"ECO_ID": 453, "COLOR": '#5AA500'},
    {"ECO_ID": 317, "COLOR": '#FDA87F'},
    {"ECO_ID": 763, "COLOR": '#A93800'},
]

def add_style_property(feature):
    color = feature.get('COLOR')
    return feature.set('style', {'color': color, 'width': 0})
ecoRegions = ecoRegions.map(add_style_property)

for update in color_updates:
    layer = ecoRegions.filter(ee.Filter.eq('ECO_ID', update['ECO_ID'])).map(
        lambda f: f.set({'COLOR': update['COLOR'], 'style': {'color': update['COLOR'], 'width': 0}})
    )
    ecoRegions = ecoRegions.filter(ee.Filter.neq('ECO_ID', update['ECO_ID'])).merge(layer)

ecoRegions = ecoRegions.style(**{'styleProperty': 'style'})

land_mask = (
    ee.Image(HGFC_ASSET_ID)
    .select("datamask")
    .eq(1)  # 1 = land, 2 = permanent water/ocean, 0 = no data
)

Map.addLayer(ecoRegions.updateMask(land_mask), {}, 'Ecoregions', 0)
Map.addLayer(site_geom, {"color": "white"}, "Test site", 1, 0.5)
Map.addLayer(grid_fc, {"color": "gray"}, "Candidate Cells")
Map.addLayer(matched_grids, {"color": "yellow"}, "Matched Cells")

Map.centerObject(grid_fc)
Map